## The Future Is Not a Feature: A Look-Ahead Bias-Free Evaluation Framework for Startup Success Prediction through Machine Learning

In [1]:
%load_ext autoreload
%autoreload 2

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl  # noqa: F401
import scipy.integrate
import yaml
from dotenv import load_dotenv

import wandb

# Used by the dataset-creation cells below, which are commented out because
# they need the original panel: without it there is nothing to rebuild.
from src.preprocessing import (  # noqa: F401
    build_full_history_dataset,
    build_windowed_dataset,
    preprocess_dataset,
)
from src.training import make_train
from src.utils import (
    clear_split_cache,
    compare_metrics,
    compute_wilcoxon_table,
    get_split,
    plot_correlation_heatmap,
    plot_shap_comparison,
    summarize_metrics,
)

# numpy 2 removed np.trapz; some of the plotting dependencies still call it.
if not hasattr(np, "trapz"):
    np.trapz = scipy.integrate.trapezoid

load_dotenv()

with open("config/config.yaml") as f:
    config = yaml.safe_load(f)

timeWindow = int(config["time_window"])
lastYear = int(config["last_year"])

# Everything prepare_splits needs, in one place: the split geometry and the
# frequency-encoding settings. The encoding is fitted per seed on the training
# split alone, so the threshold below is the only knob that decides which
# categories survive and which are pooled into "Others".
_freq = config["frequency_encoding"]
split_kwargs = dict(
    test_size=config["test_size"],
    cache_dir="tmp/splits",
    categorical_columns=_freq["columns"],
    min_frequency=_freq["min_frequency"],
    other_label=_freq["other_label"],
)

# Filled by make_train, one record per (model, tag, seed) and one SHAP entry per
# (model, tag). They live here rather than in src/ so that they accumulate
# across the experiments you run and survive an autoreload; the comparison cells
# at the bottom read them.
shap_store = {}
metrics_store = {}

/home/giulio-casti/Scrivania/UNICA/DOTTORATO/Ricerca/Startup Survival/startup-survival/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Dataset creation

The two datasets the experiments compare are built here from the **two panels**
that `build_panel.ipynb` produces, one per switch configuration. Both are derived
from PitchBook data and cannot be released; `data/processed/` already carries the
result of these cells.

| dataset | panel | features | target |
|---|---|---|---|
| `window` | switches **on** — attributes as of the row's own year | read at the age the firm first reached an early stage | does it reach the next stage **within 7 years** of that age |
| `nowindow` | switches **off** — attributes as declared at extraction | cumulated over the firm's whole observed life | does it **ever** reach the next stage |

Everything that makes the second one biased is therefore in it at once: the
attributes know the future, the features are measured after the fact, and the
label has no horizon. The two control settings further down split that sum back
into its parts.

Both panels carry the same firm-years in the same order, so the two datasets
carry the same companies and the target can be swapped between them on
`CompanyID`.

Loading of the two panels and of the university ranking

In [ ]:
# panel_timed = pl.read_csv(config["paths"]["panel_timed"], null_values=["NA"])
# panel_snapshot = pl.read_csv(config["paths"]["panel_snapshot"], null_values=["NA"])

# university_ranking_path = config["paths"]["raw_university_ranking"]

# The switches change values, never rows: if that stops holding, swapping the
# target on CompanyID between the two datasets is no longer the same firm.
# assert panel_timed.select("CompanyID", "Age").equals(panel_snapshot.select("CompanyID", "Age"))

## Dataset with time window (no look-ahead bias)
From the **timed** panel. Already available in _data/processed/dataset_window.csv_

In [ ]:
# datasetWithTimeWindow = build_windowed_dataset(panel_timed, timeWindow, lastYear)

# dataset_window = preprocess_dataset(datasetWithTimeWindow, university_ranking_path)

# dataset_window.write_csv(config["paths"]["dataset_window"])

# print("Dataset with time window saved to:", config["paths"]["dataset_window"])

## Dataset without time window (with look-ahead bias)
From the **snapshot** panel. Already available in _data/processed/dataset_nowindow.csv_

In [ ]:
# Needs the previous cell to have produced dataset_window: the firms in the
# dataset without time window must be the same ones.

# datasetWithNoTimeWindow = build_full_history_dataset(panel_snapshot, dataset_window)

# dataset_nowindow = preprocess_dataset(
#     datasetWithNoTimeWindow, university_ranking_path, flag_no_time_window=True
# )

# Same firms, same columns, same order across every experiment, or the
# comparisons are not between the same companies.
# dataset_nowindow = dataset_nowindow.filter(
#     pl.col("CompanyID").is_in(dataset_window["CompanyID"])
# )
# dataset_nowindow = dataset_nowindow.select(dataset_window.columns)

# dataset_nowindow.write_csv(config["paths"]["dataset_nowindow"])

# print("Dataset without time window saved to:", config["paths"]["dataset_nowindow"])

## Dataset selection:
(choose one)

Bias controlled experiment

In [ ]:
dataset = pd.read_csv(config["paths"]["dataset_window"])
tag = "window"

No window experiment (look-ahead bias)

In [ ]:
dataset = pd.read_csv(config["paths"]["dataset_nowindow"])
tag = "nowindow"

No Team experiment

In [ ]:
dataset = pd.read_csv(config["paths"]["dataset_window"])
dataset = dataset.drop(columns=config["ablations"]["noteam"])
tag = "noteam"

No Competitors experiment

In [ ]:
dataset = pd.read_csv(config["paths"]["dataset_window"])
dataset = dataset.drop(columns=config["ablations"]["nocompetitors"])
tag = "nocompetitors"

Label leakage only (control experiment)

The `window` and `nowindow` settings differ in three things at once: the features
— both when they are measured and whether the attributes behind them knew the
future — the definition of the target, and the base rate that follows from it.
The two cells below break that apart, so that a gap in the metrics can be
attributed to one leak rather than to their sum.

This one keeps the bias-free features, measured at `StartingAge` on the timed
panel, and takes the target of the no-window setting ("does the company ever
reach the next stage", instead of "within 7 years of `StartingAge`"). Both
processed datasets carry the same companies and the same columns, so swapping the
target on `CompanyID` is exactly the same dataset with the other label
definition.

In [ ]:
# Bias-free features + leaked target: isolates the effect of redefining the label.
_features = pd.read_csv(config["paths"]["dataset_window"]).drop(columns="Target")
_labels = pd.read_csv(config["paths"]["dataset_nowindow"])[["CompanyID", "Target"]]
dataset = _features.merge(_labels, on="CompanyID")
tag = "leaklabel"

# The base rate travels with the label definition, and no model here tunes its
# decision threshold, so it is printed as part of the experiment: AUC is the
# metric to read across the label axis, F1/precision/recall move with it.
print(f"{tag}: {len(dataset)} rows | prevalence {dataset['Target'].mean():.3f}")

Feature leakage only (control experiment)

The mirror image: the features of the no-window setting — read at the company's
last observed age, cumulated over its whole life, and built on attributes as they
were declared at extraction time — against the bias-free target, "reaches the
next stage within 7 years of `StartingAge`".

Compared with `window` it isolates the effect of measuring the features after the
fact; compared with `nowindow` it isolates the label, from the other corner of the
2x2.

In [ ]:
# Leaked features + bias-free target: isolates the effect of measuring the
# features at the end of the company's observed life instead of at StartingAge.
_features = pd.read_csv(config["paths"]["dataset_nowindow"]).drop(columns="Target")
_labels = pd.read_csv(config["paths"]["dataset_window"])[["CompanyID", "Target"]]
dataset = _features.merge(_labels, on="CompanyID")
tag = "leakfeat"

print(f"{tag}: {len(dataset)} rows | prevalence {dataset['Target'].mean():.3f}")

Feature correlation

In [ ]:
corr_matrix = plot_correlation_heatmap(dataset)

## Split, Imputation and Scaling

Run for each experiment.

One split per evaluation seed (`seeds` in _config/config.yaml_), each with its own frequency encoding, imputer and scaler fitted on its own training set. Splits are cached under _tmp/splits_, so this cell is slow only the first time per experiment.

The cache file name carries a fingerprint of the dataset columns and of the `frequency_encoding` settings, so editing the threshold in _config/config.yaml_ or regenerating the processed CSVs invalidates it on its own. Set `REBUILD_SPLITS = True` to wipe and rebuild this experiment's splits anyway.

In [ ]:
X = dataset.drop(["CompanyID", "Target"], axis=1)
y = dataset["Target"]

# One independent split per evaluation seed: 60% train, 20% validation
# (threshold tuning), 20% test. The frequency encoding, the KNN imputer and the
# scaler are each fitted on that seed's training rows alone, inside get_split,
# so no held-out row contributes to the transform later applied to it.
#
# The imputation costs minutes per split and the sweep revisits each split once
# per model, so get_split caches to tmp/splits. Running this cell fills the
# cache; afterwards each run only loads the split it needs.
REBUILD_SPLITS = False

if REBUILD_SPLITS:
    removed = clear_split_cache(cache_dir=split_kwargs["cache_dir"], tag=tag)
    print(f"cache cleared for '{tag}': {removed} file(s) removed")

for seed in config["seeds"]:
    get_split(X, y, seed, tag=tag, **split_kwargs)
    print(f"seed {seed}: split ready")

_probe = get_split(X, y, config["seeds"][0], tag=tag, **split_kwargs)
print(
    f"\nTotal: {len(y)} | Train: {len(_probe['y_train'])} | "
    f"Validation: {len(_probe['y_val'])} | Test: {len(_probe['y_test'])} "
    f"| Seeds: {config['seeds']}"
)

for column, encoding in _probe["encodings"].items():
    kept = {k: v for k, v in encoding.items() if k != split_kwargs["other_label"]}
    print(
        f"{column}: {len(kept)} categories kept, "
        f"{encoding[split_kwargs['other_label']]:.1%} pooled into "
        f"'{split_kwargs['other_label']}'"
    )

## Sweep creation

In [ ]:
entity = os.getenv("entity")
project = os.getenv("project")

# Initialize a new sweep
sweep_id = wandb.sweep(config["sweep_settings"], entity=entity, project=project)

The training function the W&B agent calls once per sweep run lives in
`src/training.py`. It builds the model for the run's `model_type` from
`src/models/`, fits it on that seed's split, logs metrics and curves, and on
`shap_seed` also computes and stores the SHAP values.

`metrics_store` and `shap_store` stay here, in the notebook, and are passed in:
they accumulate across the experiments you run, and the comparison cells at the
bottom read them.


# Experiment

## Sweep start

_tag_ is usefull for shap comparison between models, change with "nowindow" if you are running the no window experiment. 

_number_of_runs_ indicate the number of runs for the sweep. 

 - Set 70 and set a fixed model in _config/config.yaml_ (model_type) if you want to search the best hyperparams configuration. Set `seeds: &seeds [12]` in _config/config.yaml_: a single seed, and deliberately not one of the evaluation five, because several seeds would let bayes optimise the seed and report the luckiest split, and tuning on a split you later report on inflates it.

 - Set 35 if you want to test all the models with the best found configuration: set `seeds: &seeds [1, 2, 3, 4, 5]` and the grid crosses the 7 models with those 5 seeds, so each model is replicated five times. Make sure all the models are enabled in _config/config.yaml_ model_type

`seeds` is the only place a seed is written: the sweep block references it (`values: *seeds`) and the split cell prepares exactly those splits, so run the split cell again after changing it.

`svm` and `tabpfn` are part of the sweep. TabPFN (v2) runs locally: the checkpoint is downloaded to the TabPFN cache on first use, and inference needs a CUDA GPU — on CPU it is impractical at this dataset size.

The best hyperparameter configurations for each model are avaible in _config/config.yaml_ 

In [ ]:
number_of_runs = 35

train = make_train(tag, X, y, config, split_kwargs, metrics_store, shap_store)
wandb.agent(sweep_id, function=train, count=number_of_runs)

## Results of the experiment just run

One row per model, every cell the `mean ± std` across the evaluation seeds of the experiment currently selected in the *Dataset selection* cell (`tag`: `window`, `nowindow`, `noteam`, `nocompetitors`).

The std is the sample one (ddof=1) over the seeds, so a `± 0.000` cell means a single run rather than a model insensitive to the split — the `Seeds` column tells the two apart. Set `latex=True` to get the cells as `$mean \pm std$` for the paper.

In [ ]:
# One record per (model, tag, seed), so this needs that tag's 35-run grid
# (7 models x 5 seeds) to have finished.
df_results = summarize_metrics(metrics_store, tag)

seeds = config["sweep_settings"]["parameters"]["seed"]["values"]
print(f"Experiment '{tag}' — mean ± std across seeds {seeds}")
print(df_results.to_string(index=False))

## Metrics comparison across experiments

For each model, the table below compares the metrics of the `window` experiment against another experiment (`nowindow`, `nocompetitors`, `noteam`). 

Values are rounded to 2 decimals; `diff_%` is the percent change of the other experiment relative to `window` (`(other - window) / window * 100`).

Run all the experiments to populate `metrics_store` before executing the cells below.

### Comparison across window and nowindow experiments
You need to run both the "window" and "nowindow" runs before executing the following code, to ensure that shap_store and metrics_store contains the necessary data for both configurations

In [ ]:
df_cmp_nowindow = compare_metrics(metrics_store, "window", "nowindow")
print(df_cmp_nowindow.to_string(index=False))

In [ ]:
fig = plot_shap_comparison(shap_store)
plt.show()

In [ ]:
df_wilcoxon = compute_wilcoxon_table(shap_store)
print(df_wilcoxon.to_string(index=False))

### The 2x2: which leak moves the metrics

`window` is (aligned features, aligned label) and `nowindow` is (leaked, leaked), so the comparison above measures the two leaks summed together. The two tables below hold one axis fixed at a time:

- against `leaklabel`, only the target definition changes — and with it the base rate, from 0.327 to 0.410;
- against `leakfeat`, only the features change, at a constant base rate.

Read **AUC** across the label axis: it is insensitive to the base rate, while F1, precision and recall are not, since every model decides at p >= 0.5 without tuning a threshold. Run the `leaklabel` and `leakfeat` sweeps before executing these cells.

In [ ]:
df_cmp_leaklabel = compare_metrics(metrics_store, "window", "leaklabel", label_b="leaked label")
print(df_cmp_leaklabel.to_string(index=False))

In [ ]:
df_cmp_leakfeat = compare_metrics(metrics_store, "window", "leakfeat", label_b="leaked features")
print(df_cmp_leakfeat.to_string(index=False))

### Comparison across window and noteam experiments
You need to run both the "window" and "noteam" runs before executing the following code, to ensure that shap_store and metrics_store contains the necessary data for both configurations

In [ ]:
df_cmp_noteam = compare_metrics(metrics_store, "window", "noteam")
print(df_cmp_noteam.to_string(index=False))

### Comparison across window and nocompetitors experiments
You need to run both the "window" and "nocompetitors" runs before executing the following code, to ensure that shap_store and metrics_store contains the necessary data for both configurations

In [ ]:
df_cmp_nocompetitors = compare_metrics(metrics_store, "window", "nocompetitors")
print(df_cmp_nocompetitors.to_string(index=False))